# 02 数据清洗：保留风险信息并隔离测试集

清洗参数只在训练集拟合。月收入、家属人数等缺失值使用各字段训练集中位数填补，并保留质量标记；年龄 0 与逾期次数 96/98 单独标记后填补。额度使用率、债务比率的极端值暂保留原值，留待第三阶段分箱检验。

In [1]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_processing.schema import FEATURES, load_labeled_training
from src.data_processing.split import stratified_split
from src.data_processing.cleaning import CleaningRules
from src.data_processing.analysis import analyze_training

data, source = load_labeled_training(ROOT / 'data' / 'raw')
splits = stratified_split(data)
rules = CleaningRules.fit(splits.train)
train_clean = rules.transform(splits.train)
validation_clean = rules.transform(splits.validation)
test_clean = rules.transform(splits.test)
print(f'训练/验证/测试：{len(train_clean):,} / {len(validation_clean):,} / {len(test_clean):,}')
display(analyze_training(splits.train).anomalies)

训练/验证/测试：105,000 / 22,500 / 22,500


,issue,row_count,row_rate,bad_rate,handling
0,年龄为零或负值,0,0.000000,NaN,无效年龄，单独标记并用训练集中位数填补
1,逾期字段出现96/98,182,0.001733,0.543956,特殊编码，保留标记后填补
2,额度使用率大于1,2322,0.022114,0.370801,可能有业务含义，第二阶段保留原值
3,债务比率大于1,24499,0.233324,0.063962,结合收入和字段定义解读，第二阶段保留原值
4,月收入为零,1146,0.010914,0.040140,与缺失区分，第二阶段保留原值


## 训练集拟合的清洗参数

这些参数随后原样应用于验证集和测试集；两者不参与中位数计算。`qa_` 开头的字段是质量标记，不在本阶段决定是否入模。

In [2]:
import pandas as pd
display(pd.DataFrame({'字段': list(rules.medians), '训练集中位数': list(rules.medians.values())}))
display(pd.DataFrame([
    {'字段': feature, '清洗前缺失': int(splits.train[feature].isna().sum()),
     '清洗后缺失': int(train_clean[feature].isna().sum())}
    for feature in FEATURES
]))
display(train_clean.filter(regex='^qa_').sum().rename('训练集标记条数').to_frame())

,字段,训练集中位数
0,RevolvingUtilizationOfUnsecuredLines,0.153553
1,age,52.000000
2,NumberOfTime30-59DaysPastDueNotWorse,0.000000
3,DebtRatio,0.367554
4,MonthlyIncome,5382.000000
5,NumberOfOpenCreditLinesAndLoans,8.000000
6,NumberOfTimes90DaysLate,0.000000
7,NumberRealEstateLoansOrLines,1.000000
8,NumberOfTime60-89DaysPastDueNotWorse,0.000000
9,NumberOfDependents,0.000000


,字段,清洗前缺失,清洗后缺失
0,RevolvingUtilizationOfUnsecuredLines,0,0
1,age,0,0
2,NumberOfTime30-59DaysPastDueNotWorse,0,0
3,DebtRatio,0,0
4,MonthlyIncome,20723,0
5,NumberOfOpenCreditLinesAndLoans,0,0
6,NumberOfTimes90DaysLate,0,0
7,NumberRealEstateLoansOrLines,0,0
8,NumberOfTime60-89DaysPastDueNotWorse,0,0
9,NumberOfDependents,2721,0


,训练集标记条数
qa_age_invalid,0
qa_NumberOfTime30-59DaysPastDueNotWorse_special_code,17834
qa_NumberOfTimes90DaysLate_special_code,17834
qa_NumberOfTime60-89DaysPastDueNotWorse_special_code,17834
qa_late_special,182
qa_RevolvingUtilizationOfUnsecuredLines_missing,0
qa_age_missing,0
qa_NumberOfTime30-59DaysPastDueNotWorse_missing,182
qa_DebtRatio_missing,0
qa_MonthlyIncome_missing,20723


## 清洗前后与阶段边界

原始 10 个字段仍保留；处理后的逐行数据存于被 Git 忽略的 `data/processed/`。特殊值被标记而非默默删除，后续可核查其风险表现。第三阶段再建立衍生特征、正式分箱和 WOE/IV。